In [ ]:
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import f1_score, precision_score, recall_score

from google.colab import drive
drive.mount("/content/drive", force_remount=True)
OUTPUT_DIR = "/content/drive/MyDrive/RedditSentimentAnalysis/saved_models"

In [ ]:
%pip install transformers datasets accelerate scikit-learn

In [ ]:
dataset = load_dataset("go_emotions")

label_names = dataset["train"].features["labels"].feature.names
num_labels = len(label_names)

print("Number of labels:", num_labels)
print("Labels:", label_names)

In [ ]:
model_name = "cardiffnlp/twitter-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_encode(batch):
    # 1. Tokenize the text
    tokenized = tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

    # 2. Convert the list of emotion IDs into a 28-dimensional one-hot float array
    batch_labels = []
    for labels_list in batch["labels"]:
        encoded = np.zeros(num_labels, dtype=np.float32)
        encoded[labels_list] = 1.0
        batch_labels.append(encoded)

    tokenized["labels"] = batch_labels
    return tokenized

# Map and properly format to torch tensors so the Trainer can read them
dataset = dataset.map(tokenize_and_encode, batched=True, remove_columns=["text", "id"])
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
# Convert dataset labels to a NumPy array safely for fast math
labels_matrix = np.array(dataset["train"]["labels"])

pos_counts = labels_matrix.sum(axis=0)
neg_counts = labels_matrix.shape[0] - pos_counts

# Dampened weighting using square root
pos_weight_calc = np.sqrt(neg_counts / (pos_counts + 1e-6))
pos_weight = torch.tensor(pos_weight_calc, dtype=torch.float)

print("Sample weights (First 5):", pos_weight[:5])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Safely handle tuple logits
    if isinstance(logits, tuple):
        logits = logits[0]

    # Convert to probabilities using NumPy
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    return {
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=500,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, pos_weight, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = torch.nn.BCEWithLogitsLoss(
            pos_weight=self.pos_weight.to(logits.device)
        )

        # Force labels to float to prevent the casting error
        loss = loss_fct(logits, labels.float())
        return (loss, outputs) if return_outputs else loss

# Initialize the trainer
trainer = WeightedTrainer(
    pos_weight=pos_weight,
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

# Save the best model and tokenizer to your mounted Google Drive
model_path = "/content/drive/My Drive/RedditSentimentAnalysis/my_emotion_model"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)
print(f"Model successfully saved to {model_path}")

In [ ]:
# Print out the final scores on the validation set
metrics = trainer.evaluate()
print(metrics)

In [ ]:
def predict_emotions(text, threshold=0.5):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    # Apply sigmoid and convert to CPU numpy array
    probs = torch.sigmoid(logits).cpu().numpy()[0]

    return {
        label_names[i]: float(probs[i])
        for i in range(num_labels)
        if probs[i] >= threshold
    }

# Run a test prediction
test_text = "I hope thousands of Americans are contacting their congressional representatives!"
print(f"Test Sentence: '{test_text}'")
print(predict_emotions(test_text, threshold=0.3))